# Process_611336023_2022

## 資料建構作業流程
**學號：611336023**  
**檔名：Process_611336023_2022.ipynb**

本 Notebook 依照作業要求，完成 2022 年年度索引檔之資料建構流程，並輸出最終檔案 `Index_611336023_2022.csv`。


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

base_dir = Path('.')

source_file = base_dir / 'Index_611336023_2022.xlsx'
settlement_file = base_dir / '臺指選擇權_2022最後結算日.xlsx'
output_file = base_dir / 'Index_611336023_2022.csv'

print('source_file exists:', source_file.exists())
print('settlement_file exists:', settlement_file.exists())
print('output_file target:', output_file)


## 1. 讀入原始資料並確認年度範圍與資料筆數

首先讀入原始資料 `Index_611336023_2022.xlsx`，確認資料的年度範圍是否為 2022 年，並檢查資料筆數是否合理。  
此步驟的目的，是先確保原始資料本身完整，避免後續欄位建構建立在錯誤資料上。

In [ ]:
df = pd.read_excel(source_file)
df.head()


In [ ]:
df['年月日'] = pd.to_datetime(df['年月日'])

print('資料筆數：', len(df))
print('起始日期：', df['年月日'].min())
print('結束日期：', df['年月日'].max())
print('年份分布：')
print(df['年月日'].dt.year.value_counts().sort_index())


## 2. 整理日期格式並統一欄位名稱

將日期欄位統一轉為 `datetime` 格式，並將價格欄位命名為 `S0`，表示標的價格。  
此處原始資料中的 `收盤價(元)` 即作為每日標的價格使用。

In [ ]:
df = df.rename(columns={'收盤價(元)': 'S0'})
df['S0'] = pd.to_numeric(df['S0'], errors='coerce')
df = df.sort_values('年月日').reset_index(drop=True)

df.head()


In [ ]:
df.info()


## 3. 建立每日檔名 `File` 與標的價格 `S0`

### (1) File
根據每一個交易日，建立對應之每日檔名 `File`，格式為：

`OptionsDaily_YYYY_MM_DD.csv`

例如：

- 2022/01/03 → `OptionsDaily_2022_01_03.csv`
- 2022/12/30 → `OptionsDaily_2022_12_30.csv`

### (2) S0
`S0` 為每日標的價格，直接由原始資料之收盤價整理而得。

In [ ]:
df['File'] = df['年月日'].dt.strftime('OptionsDaily_%Y_%m_%d.csv')
df[['年月日', 'File', 'S0']].head()


## 4. 建立近月契約 `Contract` 與契約到期日 `ContractExpiryDate`

本作業使用 `臺指選擇權_2022最後結算日.xlsx` 建立近月契約與契約到期日。

處理方式如下：

1. 讀入最後結算日資料。
2. 僅保留 **6 碼數字** 的月契約，例如 `202201`、`202202`、`202301`。
3. 對每一個交易日，尋找**其後最近一期**的月契約最後結算日。
4. 對應之契約月份記為 `Contract`，對應之最後結算日記為 `ContractExpiryDate`。

此處採用的是「**嚴格晚於交易日**」的配對方式，因此若交易日當天恰好為某一期契約的最後結算日，則該日之後應切換至下一期月契約。

### 補充說明：2022/12/21 到 2022/12/30 這 8 筆資料的特殊情況

這 8 筆資料的 `ContractExpiryDate` 為 **2023/1/30**，原因如下：

台灣期交所在 2022/11/28 的正式公告說明，  
**2023/1/18、1/19 無交易，1/20–1/27 為除夕暨農曆春節連假休市；因此「112 年 1 月到期之國內股價指數類期貨及選擇權」的最後交易日與最後結算日，順延到 2023/1/30。**

因此，2022/12/21 到 2022/12/30 這 8 筆已對應到 **202301** 月契約，而其到期日不是一般月份常見的第三個星期三，而是依官方公告順延至 **2023/1/30**。

In [ ]:
sett = pd.read_excel(settlement_file, header=None)
sett = sett.iloc[2:].copy()
sett.columns = ['最後結算日', '契約月份', '臺指選擇權']

sett['最後結算日'] = pd.to_datetime(sett['最後結算日'])
sett['契約月份'] = sett['契約月份'].astype(str)

monthly_settlement = sett[sett['契約月份'].str.fullmatch(r'\d{6}')].copy()
monthly_settlement = monthly_settlement[['契約月份', '最後結算日']].sort_values('最後結算日').reset_index(drop=True)

monthly_settlement


In [ ]:
expiry_array = monthly_settlement['最後結算日'].to_numpy(dtype='datetime64[ns]')
contract_array = monthly_settlement['契約月份'].to_numpy()
trade_array = df['年月日'].to_numpy(dtype='datetime64[ns]')

idx = np.searchsorted(expiry_array, trade_array, side='right')

if (idx >= len(monthly_settlement)).any():
    raise ValueError('存在無法對應之交易日，請檢查最後結算日資料。')

df['Contract'] = contract_array[idx]
df['ContractExpiryDate'] = pd.to_datetime(expiry_array[idx])

df[['年月日', 'Contract', 'ContractExpiryDate']].head(15)


## 5. 計算 `Maturity`，並清楚註明距離到期天數採用日曆日

`Maturity` 定義為：**交易日距離契約到期日之剩餘天數**。

本作業採用的是 **日曆日（calendar days）**，不是交易日。  
計算方式如下：

`Maturity = (ContractExpiryDate - 年月日).days`

此外，為避免出現 0 或負值，將最小值限制為 1。

In [ ]:
df['Maturity'] = (df['ContractExpiryDate'] - df['年月日']).dt.days
df['Maturity'] = df['Maturity'].clip(lower=1).astype(int)

df[['年月日', 'ContractExpiryDate', 'Maturity']].head(15)


## 6. 補入 `Rf`，說明資料來源與合併方式

`Rf` 採用 **臺灣銀行 2022 年「定期儲蓄存款 / 一年期 / 一般金額 / 機動利率」** 作為無風險利率資料來源。

### 2022 年利率轉折點
整理後可得 2022 年的生效區間如下：

| 生效日 | Rf |
|---|---:|
| 2022-01-01 | 0.840 |
| 2022-03-21 | 1.090 |
| 2022-06-20 | 1.215 |
| 2022-09-26 | 1.340 |
| 2022-12-19 | 1.465 |

### 合併方式
由於 `Rf` 並非每日變動，而是在特定生效日調整，因此本作業採用「**逐日對齊**」方式：

- 對每一個交易日
- 尋找該日以前最近一次已生效的利率
- 將該利率填入 `Rf`

此作法可確保每一筆交易資料都對應到當日實際有效的無風險利率。

In [ ]:
rf_schedule = pd.DataFrame({
    'effective_date': pd.to_datetime([
        '2022-01-01',
        '2022-03-21',
        '2022-06-20',
        '2022-09-26',
        '2022-12-19'
    ]),
    'Rf': [0.840, 1.090, 1.215, 1.340, 1.465]
}).sort_values('effective_date').reset_index(drop=True)

rf_schedule


In [ ]:
rf_merge = pd.merge_asof(
    df[['年月日']].sort_values('年月日'),
    rf_schedule,
    left_on='年月日',
    right_on='effective_date',
    direction='backward'
)

df['Rf'] = rf_merge['Rf'].to_numpy()

if df['Rf'].isna().any():
    raise ValueError('Rf 對齊後出現缺值，請檢查生效日設定。')

df[['年月日', 'Rf']].head(20)


## 7. 輸出最終年度索引檔

最後將整理完成之欄位輸出為年度索引檔 `Index_611336023_2022.csv`。

輸出欄位包括：

- `Date`
- `S0`
- `File`
- `Contract`
- `ContractExpiryDate`
- `Maturity`
- `Rf`

其中：

- `Date` 與 `ContractExpiryDate` 轉為 `M/D/YYYY`
- `S0` 以數值格式輸出
- `Rf` 保留適當小數位數

In [ ]:
def fmt_mdy(dt):
    return dt.strftime('%-m/%-d/%Y')

def fmt_price(x):
    return f'{x:,.2f}'.rstrip('0').rstrip('.')

def fmt_rf(x):
    return f'{x:.3f}'.rstrip('0').rstrip('.')

out = df.copy()
out = out.rename(columns={'年月日': 'Date'})
out['Date'] = out['Date'].apply(fmt_mdy)
out['ContractExpiryDate'] = out['ContractExpiryDate'].apply(fmt_mdy)
out['S0'] = out['S0'].apply(fmt_price)
out['Rf'] = out['Rf'].apply(fmt_rf)

final_cols = ['Date', 'S0', 'File', 'Contract', 'ContractExpiryDate', 'Maturity', 'Rf']
out = out[final_cols]

out.head()

In [ ]:
out.to_csv(output_file, index=False, encoding='utf-8-sig')
print(f'檔案已輸出：{output_file}')


## 補充檢查

為確保資料建構結果正確，最後檢查：

1. 欄位是否完整
2. 是否存在缺值
3. `Maturity` 是否皆大於等於 1
4. 2022/12/21 到 2022/12/30 是否正確對應到 `202301 / 2023/1/30`

In [ ]:
print('列數：', len(out))
print('欄位：', list(out.columns))
print()
print('缺值統計：')
print(out.isna().sum())
print()
print('Maturity 最小值：', pd.to_numeric(out['Maturity']).min())
print()

year_end_check = out.loc[
    (pd.to_datetime(out['Date']) >= pd.Timestamp('2022-12-21')) &
    (pd.to_datetime(out['Date']) <= pd.Timestamp('2022-12-30')),
    ['Date', 'Contract', 'ContractExpiryDate', 'Maturity', 'Rf']
]
year_end_check